# Operations Reasearch: Week 6 Quiz

In [ ]:
% autoreload
%autoreloload_extad 2

2026-06-15 13:32:34.074 | INFO     | op_research.config:<module>:9 - ROOT_DIR: /Users/mmh54/Documents/github/op-research-algorithms
2026-06-15 13:32:34.075 | INFO     | op_research.config:<module>:10 - DATA_DIR: /Users/mmh54/Documents/github/op-research-algorithms/data


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Problem 1 

Fifteen jobs, each with its processing time, should be scheduled on three machines. If two jobs cannot be scheduled on the same machine, they are called conflicting jobs. Table 1 lists the job IDs, processing times, and sets of conflicting jobs. For example, we cannot schedule any pair of jobs out of jobs 2, 5, 8 on the same machine. Note that a job may have no conflicting jobs.

**Table 1.** Job IDs, processing times, and conflicting jobs.

| Job | Processing Time | Conflicting Jobs |
|:---:|:--------------:|:----------------:|
|  1  |       7        |        —         |
|  2  |       4        |      5, 8        |
|  3  |       6        |        —         |
|  4  |       9        |        —         |
|  5  |      12        |      2, 8        |
|  6  |       8        |        9         |
|  7  |      10        |       10         |
|  8  |      11        |      2, 5        |
|  9  |       8        |        6         |
| 10  |       7        |        7         |
| 11  |       6        |       15         |
| 12  |       8        |        —         |
| 13  |      15        |        —         |
| 14  |      14        |        —         |
| 15  |       3        |       11         |

We want to schedule the jobs to minimize makespan. For example, we may schedule jobs 1, 4,
7, 8, and 13 to machine 1, jobs 2, 6, 10, 11, and 14 to machine 2, and jobs 3, 5, 9, 12, and 15
to machine 3. The total processing times on the three machines are 52, 39, and 37, respectively.
The makespan is thus 52. While this is a feasible schedule, this may or may not be an optimal
schedule. When we try to improve the schedule, be careful about conflicting jobs. For example, we
cannot exchange jobs 8 and 11 (even though this reduces the makespan) because that will result
in machine 2 processing conflicting jobs 2 and 8, which is infeasible.

Formulate a linear integer program that generates a feasible schedule to minimize makespan. Then
write a computer program (e.g., using Python to invoke Gurobi Optimizer) to solve this instance
and obtain an optimal schedule. Write down the minimized makespan (i.e. the objective value of
an optimal solution). Do not have any symbol other than numeric values in your answer.

**Formulation**

The formulation is
$$
\begin{array}{rll}
    \min & \displaystyle S & \\[15pt]
    \text{s.t.} & \displaystyle \sum_{m \in M} x_{m,j} = 1 & \forall j \in J \\[15pt]
    & \displaystyle \sum_{g \in C_j} x_{m,g} \leq 1 & \forall m \in M, j \in J \\[15pt]
    & \displaystyle S \geq \sum_{j \in J} p_j x_{m,j} & \forall m \in M \\[15pt]
    & x_{m,j} \in \{0, 1\} & \forall m \in M, j \in J.
\end{array}
$$

In [4]:
import pandas as pd
from pathlib import Path
from itertools import product
#import libraries to work with pyomo
from pyomo.environ import *
from pyomo.opt import SolverFactory
from op_research.config import DATA_DIR, SOLVER


**Read Data**

In [ ]:
#read data from file
#get path of current notebook
problem01_df = pd.read_csv(DATA_DIR / "quiz06_job-processing-times.csv")
problem01_df.head()

,job,processing_time,conficting_jobs
0,1,7,NaN
1,2,4,5;8
2,3,6,NaN
3,4,9,NaN
4,5,12,2;8


Use pyomo to solve optimization model

In [ ]:
m1 = ConcreteModel()
#create sets
m1.JOBS = Set(initialize=problem01_df["job"].tolist())
#there are 3 machines
m1.MACHINES = Set(initialize=[1, 2, 3])
#------------
#decision variables
m1.x = Var(m1.JOBS, m1.MACHINES, domain=Binary, name="Assignment of job j to machine m")
m1.S = Var(domain=NonNegativeReals, name="Makespan")
#----------
#Parameters
#processing times for each job
proc_times = problem01_df.set_index("job")["processing_time"].to_dict()

#Constraints
#C1. Each job is assigned to exactly one machine
def job_assignment_rule(model, j):
    return sum(model.x[j, m] for m in model.MACHINES) == 1
m1.job_assignment = Constraint(m1.JOBS, rule=job_assignment_rule)

#C2. Define makespan constraints
def makespan_rule(model, m):
    return sum(proc_times[j] * model.x[j, m] for j in model.JOBS) <= model.S
m1.makespan_constraint = Constraint(m1.MACHINES, rule=makespan_rule)

#Note: 
#Constraint 3 is the binary nature of the decision variables, which is already defined in the variable declaration

#C3. Objective function: minimize makespan
m1.obj = Objective(expr=m1.S, sense=minimize)


In [15]:
#solve the model
solver = SolverFactory("gurobi")
results = solver.solve(m1, tee=False)
#display results
print("Optimal Makespan:", value(m1.S))
for j, m in product(m1.JOBS, m1.MACHINES):
    if value(m1.x[j, m]) > 0.05:  # binary variable, so check if it's assigned
        print(f"Job {j} is assigned to Machine {m}")

Optimal Makespan: 43.0
Job 1 is assigned to Machine 1
Job 2 is assigned to Machine 1
Job 3 is assigned to Machine 2
Job 4 is assigned to Machine 1
Job 5 is assigned to Machine 1
Job 6 is assigned to Machine 3
Job 7 is assigned to Machine 1
Job 8 is assigned to Machine 2
Job 9 is assigned to Machine 2
Job 10 is assigned to Machine 3
Job 11 is assigned to Machine 3
Job 12 is assigned to Machine 3
Job 13 is assigned to Machine 2
Job 14 is assigned to Machine 3
Job 15 is assigned to Machine 2


## Problem 2

A city is divided into *n* districts. The time (in minutes) it takes an ambulance to travel from District i to District j is denoted as $d_{i,j}$. The population of District i (in thousands) is $p_i$. An example is shown in Table 2. In this instance, we have $n= 8$ districts. We may see that, e.g., it takes 5 minutes to travel from District 2 to District 3, and there are 40,000 citizens living in District 1. 

The city has *m* ambulances and wants to locate them to mof the districts. For each district,thepopulation-weighted firefighting timeis defined as the product of the district population timesthe amount of time it takes for the closest ambulance to travel to it. The decision maker aims to locate themambulances to minimize the *maximum population-weighted firefighting* time among all districts.

As an example, suppose that m= 2, n= 8, $d_{i,j}$ and $p_i$ are provided in Table 2, and the twoambulances are located in District 1 and 8. We then know that for Districts 1, 2, and 3 the closest ambulance is in District 1 and for the remaining five districts the closest ambulance is in District 8.


**Table 2.** Travel times between districts and district populations.

| District | To 1 | To 2 | To 3 | To 4 | To 5 | To 6 | To 7 | To 8 | Population |
|:--------:|:----:|:----:|:----:|:----:|:----:|:----:|:----:|:----:|:----------:|
|    1     |  0   |  3   |  4   |  6   |  8   |  9   |  8   |  10  |     40     |
|    2     |  3   |  0   |  5   |  4   |  8   |  6   |  12  |  9   |     30     |
|    3     |  4   |  5   |  0   |  2   |  2   |  3   |  5   |  7   |     35     |
|    4     |  6   |  4   |  2   |  0   |  3   |  2   |  5   |  4   |     20     |
|    5     |  8   |  8   |  2   |  3   |  0   |  2   |  2   |  4   |     15     |
|    6     |  9   |  6   |  3   |  2   |  2   |  0   |  3   |  2   |     50     |
|    7     |  8   |  12  |  5   |  5   |  2   |  3   |  0   |  2   |     45     |
|    8     |  10  |  9   |  7   |  4   |  4   |  2   |  2   |  0   |     6

The firefighting time for the eight districts are thus 0, 3, 4, 4, 4, 2, 2, and 0 minutes, respectively. The population-weighted firefighting times may then be calculated as 0, 90, 140, 80, 60, 100, 90,and 0. The maximum among the eight districts is therefore 140.For this problem, formulate an integer program that can minimize the maximum population-weighted firefighting time among all districts. 

Then write a program to invoke a solver (e.g., write a Python program to invoke Gurobi Optimizer) to solve the above instance and find an optimalsolution for each problem. Write down the minimized maximum population-weighted firefightingtimes among all districts of the two districts that ambulances should be located in (i.e., the objectivevalue of an optimal solution). Do not have any symbol other than numeric values in your answer.

> Hint.To formulate a linear integer program for this problem, you may try to define the following set of decision variables: $x_j$ is 1 if an ambulance is located in Districtjand 0 otherwise, $y_{i,j}$ is 1 iffor Districtithe closest ambulance is located in Districtjand 0 otherwise, and $w_i$ as the distancebetween Districtiand its closest ambulance. You may then want to consider the following IP

# Formulation

$$
\begin{array}{rll}
    \min & \text{ff\_time} & \\[15pt]
    \text{s.t.} & \displaystyle \sum_{j= 1}^{n} x_{j} \leq m\\[15pt]
    & \displaystyle y_{i,j} \leq x_j & \forall i =1 \dots, n, j=1 \dots, n\\[15pt]
    & \displaystyle \sum_{j=1}^{n} y_{i,j} = 1 & \forall i=1 \dots, n \\[15pt]
    & \displaystyle \sum_{j=1}^{n} d_{i,j} * y_{i,j} \leq w_i  & \forall i=1 \dots, n\\[15pt]
    & \displaystyle p_i * w_i \leq \text{ff\_time} & \forall i=1 \dots, n \\[15pt]
    & x_j, y_{i,j} \in \{0, 1\} & \forall i =1 \dots, n, j=1 \dots, n\\[15pt]
    & w_i \geq 0 & \forall i =1 \dots, n\\[15pt]
\end{array}
$$

In [18]:
# read data from file
problem02_df = pd.read_csv(DATA_DIR / "quiz06_city-ambulance_part1.csv")
problem02_df.head(n=10)

,district_from,district_to_1,district_to_2,district_to_3,district_to_4,district_to_5,district_to_6,district_to_7,district_to_8,population
0,1,0,3,4,6,8,9,8,10,40
1,2,3,0,5,4,8,6,12,9,30
2,3,4,5,0,2,2,3,5,7,35
3,4,6,4,2,0,3,2,5,4,20
4,5,8,8,2,3,0,2,2,4,15
5,6,9,6,3,2,2,0,3,2,50
6,7,8,12,5,5,2,3,0,2,45
7,8,10,9,7,4,4,2,2,0,60


In [97]:
#create model2
m2 = ConcreteModel()
#create sets
m2.DISTRICTS = Set(initialize=problem02_df["district_from"].tolist())
#ambulances, this is manually created based on the data, we have 3 ambulances
N_AMBULANCES = 3
m2.AMBULANCES = Set(initialize=range(1, N_AMBULANCES + 1))
#------------
#decision variables
#Binary variable x_j to indicate if ambulance a is assigned to district j
m2.x = Var(m2.DISTRICTS, domain=Binary, name="Assignment of ambulance to district j")
#Auxiliary variable y_i,j to indicate if the closest ambulance for district i is located in district j
m2.y = Var(m2.DISTRICTS, m2.DISTRICTS, domain=Binary, name="Closest ambulance for district i is in district j")
#auxiliary variable to represent the distance between district i and the closest ambulance
m2.w = Var(m2.DISTRICTS, domain=NonNegativeReals, name="Distance from district i to closest ambulance")
#auxiliary variable to represent the firefighting times of all districts together, scalar, continuous, non-negative
m2.firefighting_time = Var(domain=NonNegativeReals, name="Firefighting time for all districts together")    
# #----------
#Parameters
#distance matrix between districts
# The distance matrix is constructed from the 'district_from' column to each 'district_to_x' columns, 
# where x is the district number. We will create a dictionary of dictionaries to represent this.
distance_matrix = {}
for _, row in problem02_df.iterrows():
    from_district = row["district_from"]
    distance_matrix[from_district] = {
        1: row["district_to_1"],  # convert to integer
        2: row["district_to_2"],
        3: row["district_to_3"],
        4: row["district_to_4"],
        5: row["district_to_5"],
        6: row["district_to_6"],
        7: row["district_to_7"],
        8: row["district_to_8"],
    }
#this is directly from the column population, consider that the first key elemnt should be 1 instead of zero,
#use column disctrict_from as key and population as value
population = problem02_df.set_index("district_from")["population"].to_dict()


def objective_rule(model):
    return model.firefighting_time
m2.obj = Objective(rule=objective_rule, sense=minimize)

#constraints
#C1. The sum of x_j across all districts j must be equal or less than the number of ambulances available
def ambulance_count_rule(model):
    return sum(model.x[j] for j in model.DISTRICTS) <= N_AMBULANCES
m2.ambulance_count = Constraint(rule=ambulance_count_rule)

#C2. If an ambulance is assigned to district j, then it can be the closest ambulance for district i,
# but if no ambulance is assigned to district j, then it cannot be the closest ambulance for any district i
def ambulance_assignment_rule(model, i, j):
    return model.y[i, j] <= model.x[j]
m2.ambulance_assignment = Constraint(m2.DISTRICTS, m2.DISTRICTS, 
                                     rule=ambulance_assignment_rule)

#C3. The closest ambulance for each district i must be assigned to exactly one district j
def closest_ambulance_rule(model, i):
    return sum(model.y[i, j] for j in model.DISTRICTS) == 1
m2.closest_ambulance = Constraint(m2.DISTRICTS, rule=closest_ambulance_rule)

#C4. The distance from district i to its closest ambulance must be at least the distance 
# to any ambulance assigned to a district j
def distance_to_closest_ambulance_rule(model, i, j):
    return model.w[i] >= sum(distance_matrix[i][j] * model.y[i, j] for j in model.DISTRICTS)
m2.distance_to_closest_ambulance = Constraint(m2.DISTRICTS, m2.DISTRICTS, 
                                              rule=distance_to_closest_ambulance_rule)

#C5. firefihting time is the maximum of the distance from each district to its closest 
# ambulance multiplied by the population of that district
def firefighting_time_rule(model, i):
    return model.firefighting_time >= model.w[i] * population[i] # assuming speed of ambulance is 1 unit of distance per unit of time
m2.firefighting_time_constraint = Constraint(m2.DISTRICTS, rule=firefighting_time_rule)

#solve the model
solver = SolverFactory("gurobi")
results = solver.solve(m2, tee=False)
#display results
print("Optimal Total Population-Weighted Distance:", value(m2.obj))

for j in m2.DISTRICTS:
    if value(m2.x[j]) > 0.05:  # binary variable, so check if it's assigned
        print(f"Ambulance is assigned to District {j}")


Optimal Total Population-Weighted Distance: 100.0
Ambulance is assigned to District 1
Ambulance is assigned to District 5
Ambulance is assigned to District 8


In [95]:
#multiply distance_matrix times population to see the contribution of each district to the objective function
contribution_matrix = {}
for i in distance_matrix:
    contribution_matrix[i] = {j: distance_matrix[i][j] * population[j] for j in distance_matrix[j]}
#print the contribution matrix
print("\nContribution of each district to the objective function (distance * population):")
for i in contribution_matrix:
    print(f"District {i}:")
    for j in contribution_matrix[i]:
        print(f"  To District {j}: Contribution = {contribution_matrix[i][j]}")


Contribution of each district to the objective function (distance * population):
District 1:
  To District 1: Contribution = 0
  To District 2: Contribution = 90
  To District 3: Contribution = 140
  To District 4: Contribution = 120
  To District 5: Contribution = 120
  To District 6: Contribution = 450
  To District 7: Contribution = 360
  To District 8: Contribution = 600
District 2:
  To District 1: Contribution = 120
  To District 2: Contribution = 0
  To District 3: Contribution = 175
  To District 4: Contribution = 80
  To District 5: Contribution = 120
  To District 6: Contribution = 300
  To District 7: Contribution = 540
  To District 8: Contribution = 540
District 3:
  To District 1: Contribution = 160
  To District 2: Contribution = 150
  To District 3: Contribution = 0
  To District 4: Contribution = 40
  To District 5: Contribution = 30
  To District 6: Contribution = 150
  To District 7: Contribution = 225
  To District 8: Contribution = 420
District 4:
  To District 1:

## Problem 3

Continue from the previous question. For any value of $m$, consider the following heuristic algorithm which runs $m$ iterations. In each iteration, we locate an ambulance in a district that (1) currently does not have an ambulance, and (2) may minimize the maximum population-weighted firefighting times among all districts. If there are multiple districts satisfying these two conditions, pick the one with the smallest district ID among them. We then proceed to the next iteration to look for the next district to locate an ambulance.

Consider a tiny example with n= 4, m= 2, and $d_{i,j}$ and $p_j$ are provided in Table 3. To locate the first ambulance, we examine the maximum population-weighted firefighting times of locating an ambulance in Districts 1, 2, 3, and 4 as 140, 175, 160, and 240, respectively. We will choose District 1.  To locate the second ambulance, we take the ambulance in District 1 as given and examine the maximum population-weighted firefighting times of locating an ambulance in Districts 2, 3, and 4 as 175, 150, and 240, respectively. We will choose District 3. The final objective value of locating two ambulances in Districts 1 and 3 is 90 (which is 30×3 for District 2)

**Table 3.** Travel times between districts (in minutes) and district populations (in thousands).

| District | To 1 | To 2 | To 3 | To 4 | Population |
|:--------:|:----:|:----:|:----:|:----:|:----------:|
|    1     |  0   |  3   |  4   |  1   |     40     |
|    2     |  3   |  0   |  5   |  8   |     30     |
|    3     |  4   |  5   |  0   |  1   |     35     |
|    4     |  1   |  8   |  1   |  0   |      5     |



Coming back to the instance provided in Table 2 with n= 8. Now let m= 3. Use the heuristic algorithm introduced above to generate a feasible solution. Then use the program you wrote forQuestion 2 to generate an optimal solution. Write down the absolute optimality gap between thetwo solutions (i.e., the difference between the two objective values). Do not have any symbol other than numeric values in your answer.

In [90]:
#read data from file
problem03_df = pd.read_csv(DATA_DIR / "quiz06_city-ambulance_part2.csv")
problem03_df.head(n=10)

,district_from,distric_to_1,distric_to_2,distric_to_3,distric_to_4,population
0,1,0,3,4,1,40
1,2,3,0,5,8,30
2,3,4,5,0,1,35
3,4,1,8,1,0,5


In [100]:
#create model2
m3 = ConcreteModel()
#create sets
m3.DISTRICTS = Set(initialize=problem03_df["district_from"].tolist())
#ambulances, this is manually created based on the data, we have 3 ambulances
N_AMBULANCES = 3
m3.AMBULANCES = Set(initialize=range(1, N_AMBULANCES + 1))
#------------
#decision variables
#Binary variable x_j to indicate if ambulance a is assigned to district j
m3.x = Var(m3.DISTRICTS, domain=Binary, name="Assignment of ambulance to district j")
#Auxiliary variable y_i,j to indicate if the closest ambulance for district i is located in district j
m3.y = Var(m3.DISTRICTS, m3.DISTRICTS, domain=Binary, name="Closest ambulance for district i is in district j")
#auxiliary variable to represent the distance between district i and the closest ambulance
m3.w = Var(m3.DISTRICTS, domain=NonNegativeReals, name="Distance from district i to closest ambulance")
#auxiliary variable to represent the firefighting times of all districts together, scalar, continuous, non-negative
m3.firefighting_time = Var(domain=NonNegativeReals, name="Firefighting time for all districts together")    
# #----------
#Parameters
#distance matrix between districts
# The distance matrix is constructed from the 'district_from' column to each 'district_to_x' columns, 
# where x is the district number. We will create a dictionary of dictionaries to represent this.
distance_matrix = {}
for _, row in problem03_df.iterrows():
    from_district = row["district_from"]
    distance_matrix[from_district] = {
        1: row["distric_to_1"],  # convert to integer
        2: row["distric_to_2"],
        3: row["distric_to_3"],
        4: row["distric_to_4"]
    }
#this is directly from the column population, consider that the first key elemnt should be 1 instead of zero,
#use column disctrict_from as key and population as value
population = problem03_df.set_index("district_from")["population"].to_dict()


def objective_rule(model):
    return model.firefighting_time
m3.obj = Objective(rule=objective_rule, sense=minimize)

#constraints
#C1. The sum of x_j across all districts j must be equal or less than the number of ambulances available
def ambulance_count_rule(model):
    return sum(model.x[j] for j in model.DISTRICTS) <= N_AMBULANCES
m3.ambulance_count = Constraint(rule=ambulance_count_rule)

#C2. If an ambulance is assigned to district j, then it can be the closest ambulance for district i,
# but if no ambulance is assigned to district j, then it cannot be the closest ambulance for any district i
def ambulance_assignment_rule(model, i, j):
    return model.y[i, j] <= model.x[j]
m3.ambulance_assignment = Constraint(m3.DISTRICTS, m3.DISTRICTS, 
                                     rule=ambulance_assignment_rule)

#C3. The closest ambulance for each district i must be assigned to exactly one district j
def closest_ambulance_rule(model, i):
    return sum(model.y[i, j] for j in model.DISTRICTS) == 1
m3.closest_ambulance = Constraint(m3.DISTRICTS, rule=closest_ambulance_rule)

#C4. The distance from district i to its closest ambulance must be at least the distance 
# to any ambulance assigned to a district j
def distance_to_closest_ambulance_rule(model, i, j):
    return model.w[i] >= sum(distance_matrix[i][j] * model.y[i, j] for j in model.DISTRICTS)
m3.distance_to_closest_ambulance = Constraint(m3.DISTRICTS, m3.DISTRICTS, 
                                              rule=distance_to_closest_ambulance_rule)

#C5. firefihting time is the maximum of the distance from each district to its closest 
# ambulance multiplied by the population of that district
def firefighting_time_rule(model, i):
    return model.firefighting_time >= model.w[i] * population[i] # assuming speed of ambulance is 1 unit of distance per unit of time
m3.firefighting_time_constraint = Constraint(m3.DISTRICTS, rule=firefighting_time_rule)

#solve the model
solver = SolverFactory("gurobi")
results = solver.solve(m3, tee=False)
#display results
print("Optimal Total Population-Weighted Distance:", value(m3.obj))

for j in m3.DISTRICTS:
    if value(m3.x[j]) > 0.05:  # binary variable, so check if it's assigned
        print(f"Ambulance is assigned to District {j}")

Optimal Total Population-Weighted Distance: 5.0
Ambulance is assigned to District 1
Ambulance is assigned to District 2
Ambulance is assigned to District 3
